# Churn Distribution Analysis Notebook

## Overview
This notebook analyzes churn distribution across different feature types to understand which factors correlate with customer churn.

### Analysis Agenda:
1. **Overall Churn Distribution** - Baseline churn rate and patterns
2. **Demographic Features** - Age, gender, marital status, dependents
3. **Service Features (Binary)** - Yes/No service subscriptions
4. **Service Features (Continuous)** - Numerical service metrics  
5. **Service Features (Ordinal)** - Categorical features with order


### Key Objectives:
- Identify statistically significant differences between churn and non-churn groups
- Visualize distributions for each feature type
- Prepare insights for downstream modeling stages
- Document feature importance for churn prediction

### Methodology:
- Statistical tests (t-tests for continuous, proportion tests for categorical)
- Visualization (bar charts, box plots, histograms)
- Consistent color scheme for easy interpretation

---

*Note: All data has been preprocessed and is ready for analysis.*

In [ ]:
# =============================================================================
# CHURN ANALYSIS - ORGANIZED BY FEATURE TYPE (FIXED)
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set consistent color scheme
CHURN_COLOR = '#c0392b'
NO_CHURN_COLOR = '#27ae60'
COLORS = [NO_CHURN_COLOR, CHURN_COLOR]


## 2. Data Loading and Preparation

This section loads the preprocessed data from Stage 1 and prepares it for analysis by creating separate datasets for churned and retained customers.

**Process:**
1. Load the preprocessed CSV file
2. Display key statistics about the dataset
3. Split data into churn (1) and no-churn (0) groups for comparison
4. Verify data quality and completeness

**Purpose:** Establish the foundation for all subsequent analyses by ensuring data is properly loaded and segmented.

In [ ]:
# ============================================================================
# 2. LOAD PREPROCESSED DATA
# ============================================================================
print("📁 LOADING PREPROCESSED DATA")
print("=" * 60)

# Load the preprocessed data from Stage 1
data = pd.read_csv('/content/sample_data/churn_data_preprocessed.csv')

print(f"✅ Data loaded successfully!")
print(f"   Shape: {data.shape}")
print(f"   Features: {data.shape[1]}")
print(f"   Samples: {data.shape[0]}")
print(f"   Overall Churn Rate: {data['Churn'].mean():.2%}")
print(f"   Missing Values: {data.isnull().sum().sum()}")
print("-" * 60)
print("📋 First few rows:")
print(data.head())
print("=" * 60)

# Section 3: Churn Analysis - Organized by Feature Type

## Purpose
Establish the baseline churn rate and analyze churn patterns across different feature categories to identify key drivers of customer attrition.

## 3A. Overall Churn Distribution

### Purpose
Establish the baseline churn rate and visualize the overall distribution of churned vs retained customers.

### Key Questions Answered
- What is the overall churn rate in our dataset?
- How is churn distributed across the customer base?
- What visualizations best represent the churn imbalance?

### Insights to Gather
- Baseline churn percentage for comparison with feature-specific rates
- Understanding of the class imbalance in our dataset
- Foundation for evaluating feature importance relative to overall churn

### Key Insight
The overall churn rate is **26.5%**, indicating a moderate class imbalance where retained customers (73.5%, over 5,000) heavily outnumber churned customers (under 2,000). This baseline rate of 26.5% serves as the threshold: any customer segment with a higher churn rate is considered high-risk, and the imbalance must be accounted for in the predictive modeling phase.


In [ ]:
# =============================================================================
# Section 3: CHURN ANALYSIS - ORGANIZED BY FEATURE TYPE
# =============================================================================

# Split churned vs retained
churn_data = data[data['Churn'] == 1]
no_churn_data = data[data['Churn'] == 0]

# ---------------------------
# SECTION 1: OVERALL CHURN
# ---------------------------
churn_counts = data['Churn'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Overall Churn Distribution', fontsize=16, fontweight='bold')

# Pie chart
axes[0].pie(churn_counts.values, labels=['No Churn', 'Churn'], autopct='%1.1f%%',
            startangle=90, colors=COLORS, explode=(0, 0.1), shadow=True)
axes[0].set_title('Churn Distribution (Pie Chart)')

# Bar chart
churn_counts.plot(kind='bar', ax=axes[1], color=COLORS)
axes[1].set_title('Churn Distribution (Bar Chart)')
axes[1].set_xticks(range(len(churn_counts)))  # FIX: Add set_xticks first
axes[1].set_xticklabels(['No Churn', 'Churn'], rotation=0)
axes[1].set_ylabel('Number of Customers')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 3B. Demographic Features (Categorical)

### Purpose
Analyze how categorical demographic characteristics (Gender, Marital Status) relate to customer churn.

### Key Questions Answered
- Does gender influence churn rates?
- Does marital status affect customer retention?
- Are there significant differences in churn across demographic segments?

### Insights to Gather
- Churn rates by gender
- Churn rates by marital status
- Statistical significance of demographic factors
- Comparison to baseline churn rate (26.5%)

### Key Features Analyzed
- **Gender**: Female vs Male
- **Married**: Not Married vs Married


In [ ]:
# ---------------------------
# SECTION 3B: DEMOGRAPHIC FEATURES (CATEGORICAL)
# ---------------------------
demographic_categorical = {'Gender': {0: 'Female', 1: 'Male'},
                           'Married': {0: 'Not Married', 1: 'Married'}}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Demographic Features: Churn Distribution', fontsize=16, fontweight='bold')

for i, (feature, labels) in enumerate(demographic_categorical.items()):
    if feature in data.columns:
        churn_by_feature = data.groupby(feature)['Churn'].agg(['sum', 'count', 'mean'])
        df_plot = pd.DataFrame({
            'No Churn': [no_churn_data[feature].value_counts().get(idx, 0) for idx in labels.keys()],
            'Churn': [churn_data[feature].value_counts().get(idx, 0) for idx in labels.keys()]
        }, index=list(labels.values()))
        df_plot.plot(kind='bar', ax=axes[i], color=COLORS)
        axes[i].set_title(f'{feature} Distribution')
        axes[i].set_ylabel('Number of Customers')
        axes[i].set_xticks(range(len(labels)))  # FIX: Add set_xticks
        axes[i].set_xticklabels(list(labels.values()), rotation=0)
        axes[i].legend(title='Status')
        axes[i].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 3C. Demographic Features (Continuous)

### Purpose
Examine how continuous demographic variables (Age, Number of Dependents, Number of Referrals, Population density) correlate with churn behavior.

### Key Questions Answered
- How does age distribution differ between churned and retained customers?
- Does having dependents affect churn likelihood?
- Do referrals indicate customer satisfaction and lower churn?
- Does population density in customer location impact churn?

### Insights to Gather
- Mean/median comparisons between churned and non-churned customers
- Distribution differences for continuous demographic features
- Statistical significance tests (t-tests or Mann-Whitney U)
- Identification of demographic thresholds associated with higher churn

### Key Features Analyzed
- **Age**
- **NumberOfDependents**
- **NumberOfReferrals**
- **Population**


In [ ]:
# ---------------------------
# SECTION 3C: DEMOGRAPHIC FEATURES (CONTINUOUS)
# ---------------------------

print("=" * 80)
print("SECTION 3: DEMOGRAPHIC FEATURES (CONTINUOUS)")
print("=" * 80)

demographic_continuous = ['Age', 'NumberofDependents', 'NumberofReferrals', 'Population']

# --- 1. Statistical Analysis ---
print("\n📊 STATISTICAL COMPARISON (DEMOGRAPHIC CONTINUOUS FEATURES):")
print("-" * 100)
print(f"{'Feature':<30} {'Churn Mean':<15} {'No-Churn Mean':<15} {'Difference':<15} {'p-value'}")
print("-" * 100)

for col in demographic_continuous:
    if col in data.columns:
        churn_vals = churn_data[col].dropna()
        nochurn_vals = no_churn_data[col].dropna()

        if len(churn_vals) > 1 and len(nochurn_vals) > 1:
            t_stat, p_value = stats.ttest_ind(churn_vals, nochurn_vals, equal_var=False)
            churn_mean = churn_vals.mean()
            nochurn_mean = nochurn_vals.mean()
            difference = churn_mean - nochurn_mean
            significance = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'

            # Formatted printout matching Section 5A
            print(f"{col:<30} {churn_mean:<15.2f} {nochurn_mean:<15.2f} {difference:<15.2f} {p_value:.6f} {significance}")


# --- 2. Visualization: Distribution (Histogram with KDE) ---
n_features = len(demographic_continuous)
n_cols = 2
n_rows = (n_features + n_cols - 1) // n_cols

fig_dist, axes_dist = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 5))
fig_dist.suptitle('Demographic Features: Churn Distribution (Histogram/KDE)', fontsize=16, fontweight='bold')

# Flatten axes array for easy iteration if n_rows > 1
axes_dist = axes_dist.flatten() if n_rows > 1 or n_cols > 1 else np.array([axes_dist])

for idx, col in enumerate(demographic_continuous):
    if col in data.columns:
        ax = axes_dist[idx]

        # Plotting No Churn distribution
        sns.histplot(data=no_churn_data, x=col, label='No Churn', alpha=0.6,
                     ax=ax, color=NO_CHURN_COLOR, kde=True, bins=30,
                     line_kws={'linewidth': 1.5})

        # Plotting Churn distribution
        sns.histplot(data=churn_data, x=col, label='Churn', alpha=0.6,
                     ax=ax, color=CHURN_COLOR, kde=True, bins=30,
                     line_kws={'linewidth': 1.5})

        ax.set_title(f'{col} Distribution')
        ax.set_xlabel(col)
        ax.set_ylabel('Frequency')
        ax.legend(loc='upper right')
        ax.grid(axis='y', alpha=0.3)

# Hide any empty subplots
for idx in range(len(demographic_continuous), n_rows * n_cols):
    if idx < len(axes_dist):
        axes_dist[idx].axis('off')

plt.tight_layout()
plt.show()


# --- 3. Visualization: Boxplots (The existing, fixed code block) ---
fig_box, axes_box = plt.subplots(2, 2, figsize=(14, 10))
fig_box.suptitle('Demographic Continuous Features: Box Plots', fontsize=16, fontweight='bold')

for idx, col in enumerate(demographic_continuous):
    if col in data.columns:
        ax = axes_box[idx//2, idx%2]
        sns.boxplot(data=data, x='Churn', y=col, hue='Churn', ax=ax, palette=COLORS, legend=False)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['No Churn', 'Churn'])
        ax.set_title(f'{col} by Churn Status')
        ax.grid(axis='y', alpha=0.3)

# Hide unused axes
if len(demographic_continuous) < 4:
    axes_box[1,1].axis('off')

plt.tight_layout()
plt.show()

### Key Findings

*Age Distribution: The age distribution shows relatively similar patterns between churned and retained customers, with a notable concentration of younger customers (ages 20-25) in both groups. Churned customers show a slight uptick in the 75-80 age range, suggesting potential vulnerability among senior customers. The distributions are fairly balanced across middle-age ranges (30-65), indicating age alone may not be a primary churn driver.*

*Number of Dependents: A stark pattern emerges - the vast majority of customers (both churned and retained) have zero dependents, with exponentially decreasing frequency as dependent count increases. The proportional representation of churned vs retained customers appears consistent across all dependent counts, suggesting this feature may have limited discriminatory power for churn prediction.*

*Number of Referrals: This feature shows the strongest differentiation pattern. Most customers have zero referrals, and within this group, churned customers are heavily represented. As referral count increases (1-10+), the proportion of churned customers decreases dramatically. This indicates that referral behavior is strongly associated with customer loyalty and satisfaction - customers who refer others are significantly less likely to churn.*

*Population Distribution: Both churned and retained customers are concentrated in lower population density areas (0-20,000), with frequency decreasing as population increases. The distributions between churned and retained customers are proportionally similar across all population ranges, suggesting population density has minimal impact on churn behavior. The slight concentration of both groups in lower-density areas may simply reflect the dataset's geographic composition.*

## 3D. Service Features (Categorical - Binary)

### Purpose
Analyze binary service adoption features to identify which services are associated with higher or lower churn rates.

### Key Questions Answered
- Which services show the strongest correlation with churn?
- Do customers with internet service churn more/less?
- How do add-on services affect retention?
- What is the churn rate difference between paperless and paper billing?

### Insights to Gather
- Churn rates for each binary service feature
- Comparison to baseline (26.5%) to identify high-risk services
- Service combinations that may indicate churn risk
- Most protective vs most risky service features

### Key Features Analyzed
Binary service features (Yes/No):
- **InternetService**: No Internet vs Has Internet
- **PhoneService**: No Phone vs Has Phone
- **MultipleLines**: No/No Phone vs Multiple Lines
- **OnlineSecurity**: No vs Yes
- **OnlineBackup**: No vs Yes
- **DeviceProtectionPlan**: No vs Yes
- **PremiumTechSupport**: No vs Yes
- **StreamingTV**: No vs Yes
- **StreamingMovies**: No vs Yes
- **StreamingMusic**: No vs Yes
- **UnlimitedData**: No vs Yes
- **PaperlessBilling**: Paper Billing vs Paperless Billing

In [ ]:
# =============================================================================
# SECTION 3D: SERVICE FEATURES (CATEGORICAL - BINARY)
# =============================================================================

print("=" * 80)
print("SECTION 4: SERVICE FEATURES - CATEGORICAL (BINARY)")
print("=" * 80)

# Statistical Analysis
print("\n📊 CHURN RATES BY BINARY SERVICE FEATURES:")
print("-" * 70)

binary_service_features = {
    'InternetService': {0: 'No Internet', 1: 'Has Internet'},
    'PhoneService': {0: 'No Phone', 1: 'Has Phone'},
    'MultipleLines': {0: 'No Multiple Lines', 1: 'Has Multiple Lines'},
    'OnlineSecurity': {0: 'No', 1: 'Yes'},
    'OnlineBackup': {0: 'No', 1: 'Yes'},
    'DeviceProtectionPlan': {0: 'No', 1: 'Yes'},
    'PremiumTechSupport': {0: 'No', 1: 'Yes'},
    'StreamingTV': {0: 'No', 1: 'Yes'},
    'StreamingMovies': {0: 'No', 1: 'Yes'},
    'StreamingMusic': {0: 'No', 1: 'Yes'},
    'UnlimitedData': {0: 'No', 1: 'Yes'},
    'PaperlessBilling': {0: 'Paper Billing', 1: 'Paperless Billing'}
}

# Filter to features that exist in our data
existing_features = {k: v for k, v in binary_service_features.items() if k in data.columns}

print(f"Analyzing {len(existing_features)} binary service features...\n")

for feature, labels in existing_features.items():
    churn_by_feature = data.groupby(feature)['Churn'].agg(['sum', 'count', 'mean'])
    print(f"\n{feature}:")
    for idx, row in churn_by_feature.iterrows():
        label = labels.get(idx, idx)
        print(f"   {label}: {row['mean']:.1%} churn rate ({int(row['sum'])}/{int(row['count'])} customers)")

# Visualization: Binary Service Features
n_features = len(existing_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols  # Ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 5))
fig.suptitle('Binary Service Features: Churn Distribution', fontsize=16, fontweight='bold')

# Flatten axes for easier indexing
if n_rows == 1:
    axes = axes.reshape(1, -1)
elif n_cols == 1:
    axes = axes.reshape(-1, 1)
axes_flat = axes.flatten()

for idx, (feature, labels) in enumerate(existing_features.items()):
    if idx < len(axes_flat):
        ax = axes_flat[idx]

        # Get counts
        feature_churn = churn_data[feature].value_counts().sort_index()
        feature_nochurn = no_churn_data[feature].value_counts().sort_index()

        # Create DataFrame - handle missing indices
        max_idx = 1  # Binary features are 0 and 1
        feature_df = pd.DataFrame({
            'No Churn': [feature_nochurn.get(i, 0) for i in range(max_idx + 1)],
            'Churn': [feature_churn.get(i, 0) for i in range(max_idx + 1)]
        }, index=[labels.get(i, f'Value {i}') for i in range(max_idx + 1)])

        # Plot
        feature_df.plot(kind='bar', ax=ax, color=COLORS)
        ax.set_title(f'{feature}')
        ax.set_xlabel(feature)
        ax.set_ylabel('Number of Customers')
        ax.set_xticks(range(len(feature_df.index)))
        ax.set_xticklabels(feature_df.index, rotation=45, ha='right')
        ax.legend(title='Status')
        ax.grid(axis='y', alpha=0.3)

# Hide any empty subplots
for idx in range(len(existing_features), n_rows * n_cols):
    axes_flat[idx].axis('off')

plt.tight_layout()
plt.show()

print("\n✅ Section 3D Complete\n")

### Key Findings

***Internet Service**: Shows the strongest churn differential of all binary features. Customers without internet service (1,400 total) show minimal churn, while those with internet service (5,200 total) exhibit substantial churn (1,700 churned). This suggests internet service customers represent the primary at-risk segment - possibly due to competitive markets, service quality issues, or pricing sensitivity.*

***Phone Service**: Displays relatively low churn rates across both categories. The small "No Phone" segment (700 total) shows very minimal churn (200), while the much larger "Has Phone" segment (4,600 total) maintains a moderate churn rate (1,600). Phone service appears to be a stable, less contentious product line.*

***Multiple Lines**: Among phone service customers, those with multiple lines (2,100 total, 850 churned) show higher absolute churn numbers compared to single line customers, though this may be proportional to their representation in the customer base.*

***Security & Protection Services (OnlineSecurity, OnlineBackup, DeviceProtectionPlan)**: These three features show remarkably similar patterns - customers WITHOUT these services consistently show 3-5x higher churn counts than those WITH the services. This is a critical finding: protective/security add-ons are strongly associated with retention. Customers who invest in these services demonstrate higher engagement and lower churn likelihood.*

***Premium Tech Support**: Exhibits one of the clearest protective effects. Customers without premium support (3,400 total, 1,500 churned) have significantly higher churn than those with it (1,700 total, 300 churned). The churn rate appears much lower among premium support subscribers, suggesting this service either attracts more committed customers or effectively prevents churn through better service experiences.*

***Streaming Services (TV, Movies, Music)**: All three streaming features show similar patterns - customers with streaming services have moderate-to-high absolute churn numbers, but this appears roughly proportional to their larger customer base. Customers without streaming services show lower absolute churn but also represent smaller segments. Streaming adoption doesn't appear to be a strong protective factor against churn.*

***Unlimited Data**: Shows a distinctive pattern. The smaller "No Unlimited Data" segment (2,000 total) has very low churn (350), while the larger "Has Unlimited Data" segment (3,500 total) shows substantial churn (1,450). This suggests unlimited data plans may be associated with higher-usage, more price-sensitive customers who are more likely to shop around for better deals.*

***Paperless Billing**: Customers with paper billing (2,400 total, 450 churned) show significantly lower churn rates compared to paperless billing customers (2,900 total, 1,400 churned). This is counterintuitive - paperless billing customers, despite being potentially more tech-savvy and engaged, show higher churn. This may indicate that paperless customers are more comfortable switching providers online, while paper billing customers demonstrate more traditional loyalty patterns.*

**Critical Pattern Identified**: Security and support services (OnlineSecurity, OnlineBackup, DeviceProtectionPlan, PremiumTechSupport) act as strong retention anchors. The "protective service bundle" effect suggests that customers who adopt multiple add-on services are significantly more invested and less likely to churn, while "bare-bones" internet-only customers represent the highest-risk segment.



## 3E. Service Features (Continuous)

### Purpose
Examine continuous service-related metrics to understand how usage patterns and financial factors relate to churn.

### Key Questions Answered
- How does tenure relate to churn probability?
- Do customers with higher monthly charges churn more?
- Does total spend indicate loyalty?
- How do various charge types correlate with churn?

### Insights to Gather
- Distribution comparisons between churned and retained for each feature
- Mean/median differences and statistical significance
- Identification of financial thresholds associated with churn risk
- Tenure patterns that indicate stable vs at-risk customers

### Key Features Analyzed
- **TenureInMonths**
- **MonthlyCharge**
- **TotalCharges**
- **AvgMonthlyLongDistanceCharges**
- **AvgMonthlyGBDownload**
- **TotalRefunds**
- **TotalExtraDataCharges**
- **TotalRevenue**

In [ ]:
# =============================================================================
# SECTION 3E: SERVICE FEATURES (CONTINUOUS)
# =============================================================================

print("=" * 80)
print("SECTION 3E: SERVICE FEATURES - CONTINUOUS")
print("=" * 80)

continuous_features = [
    'TenureinMonths',
    'MonthlyCharge',
    'TotalCharges',
    'AvgMonthlyLongDistanceCharges',
    'AvgMonthlyGBDownload',
    'TotalRefunds',
    'TotalExtraDataCharges',
    'TotalRevenue'
]

# Filter to existing columns only
continuous_features = [col for col in continuous_features if col in data.columns]

# --- 1. Statistical Analysis ---
print("\n📊 STATISTICAL COMPARISON (CONTINUOUS FEATURES):")
print("-" * 100)
print(f"{'Feature':<30} {'Churn Mean':<15} {'No-Churn Mean':<15} {'Difference':<15} {'p-value'}")
print("-" * 100)

for col in continuous_features:
    churn_vals = churn_data[col].dropna()
    nochurn_vals = no_churn_data[col].dropna()

    if len(churn_vals) > 1 and len(nochurn_vals) > 1:
        # T-test for independent samples
        t_stat, p_value = stats.ttest_ind(churn_vals, nochurn_vals, equal_var=False)
        churn_mean = churn_vals.mean()
        nochurn_mean = nochurn_vals.mean()
        difference = churn_mean - nochurn_mean
        significance = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'

        print(f"{col:<30} {churn_mean:<15.2f} {nochurn_mean:<15.2f} {difference:<15.2f} {p_value:.6f} {significance}")

# --- 2. Visualization: Distribution (Histogram with KDE) ---
n_features = len(continuous_features)
n_cols = 3
n_rows_dist = (n_features + n_cols - 1) // n_cols # Ceiling division for rows

fig_dist, axes_dist = plt.subplots(n_rows_dist, n_cols, figsize=(18, n_rows_dist * 5))
fig_dist.suptitle('Continuous Service Features: Churn Distribution (Histogram/KDE)', fontsize=16, fontweight='bold')

# Ensure axes is a 2D array even for a single row
if n_rows_dist == 1:
    axes_dist = axes_dist.reshape(1, -1)
elif n_cols == 1:
    axes_dist = axes_dist.reshape(-1, 1)

for idx, col in enumerate(continuous_features):
    row = idx // n_cols
    col_pos = idx % n_cols

    # Plotting No Churn distribution
    sns.histplot(data=no_churn_data, x=col, label='No Churn', alpha=0.6,
                 ax=axes_dist[row, col_pos], color=NO_CHURN_COLOR, kde=True, bins=30,
                 line_kws={'linewidth': 1.5}) # Added line_kws for slightly thicker KDE line

    # Plotting Churn distribution
    sns.histplot(data=churn_data, x=col, label='Churn', alpha=0.6,
                 ax=axes_dist[row, col_pos], color=CHURN_COLOR, kde=True, bins=30,
                 line_kws={'linewidth': 1.5})

    axes_dist[row, col_pos].set_title(f'{col} Distribution')
    axes_dist[row, col_pos].set_xlabel(col)
    axes_dist[row, col_pos].set_ylabel('Frequency')
    axes_dist[row, col_pos].legend(loc='upper right')
    axes_dist[row, col_pos].grid(axis='y', alpha=0.3)

# Hide any empty subplots
for idx in range(len(continuous_features), n_rows_dist * n_cols):
    row = idx // n_cols
    col_pos = idx % n_cols
    axes_dist[row, col_pos].axis('off')

plt.tight_layout()
plt.show()


# --- 3. Visualization: Boxplots (The existing code block) ---
n_features = len(continuous_features)
n_cols = 3
n_rows_box = (n_features + n_cols - 1) // n_cols # Ceiling division for rows

fig_box, axes_box = plt.subplots(n_rows_box, n_cols, figsize=(18, n_rows_box * 5))
fig_box.suptitle('Continuous Service Features: Box Plots', fontsize=16, fontweight='bold')

if n_rows_box == 1:
    axes_box = axes_box.reshape(1, -1)
elif n_cols == 1:
    axes_box = axes_box.reshape(-1, 1)

for idx, col in enumerate(continuous_features):
    row = idx // n_cols
    col_pos = idx % n_cols

    sns.boxplot(data=data, x='Churn', y=col, hue='Churn', ax=axes_box[row, col_pos],
                palette=COLORS, legend=False)
    axes_box[row, col_pos].set_title(f'{col} by Churn Status')
    axes_box[row, col_pos].set_xticks([0, 1])
    axes_box[row, col_pos].set_xticklabels(['No Churn', 'Churn'])
    axes_box[row, col_pos].set_xlabel('Churn Status')
    axes_box[row, col_pos].set_ylabel(col)
    axes_box[row, col_pos].grid(axis='y', alpha=0.3)

# Hide any empty subplots
for idx in range(len(continuous_features), n_rows_box * n_cols):
    row = idx // n_cols
    col_pos = idx % n_cols
    axes_box[row, col_pos].axis('off')

plt.tight_layout()
plt.show()

print("\n✅ Section 3E Complete\n")

### Key Findings

***Tenure in Months**: Reveals the most dramatic churn pattern. Churned customers are overwhelmingly concentrated in the 0-10 month range, with the highest frequency at 0-5 months (~ 600 churned). This represents a critical "danger zone" - new customers who haven't yet developed loyalty. In stark contrast, retained customers show a strong accumulation at 70+ months (600 retained), indicating that customers who survive the early period become highly stable. The middle tenure ranges (10-65 months) show relatively balanced distributions with lower churn proportions, suggesting a "survivorship curve" where early churn risk decreases substantially after the first year.*

***Monthly Charge**: Displays a clear bimodal distribution. Churned customers are heavily concentrated in two ranges: low charges ($0-30) with ~600 frequency, and mid-to-high charges ($65-100) with sustained churn throughout. Retained customers show peaks in the $20-40 range (~550-600 frequency) and are more evenly distributed across higher price points. The pattern suggests two distinct churn profiles: (1) low-value customers who leave quickly, and (2) higher-paying customers who may be experiencing sticker shock or finding better competitive offers. The $65-100 range appears to be a particularly sensitive price point where churn risk elevates.*

***Total Charges**: Shows an exponential decay pattern where both churned and retained customers are concentrated in the lower total spend ranges ($0-2000), with frequency dropping off sharply as total charges increase. Critically, churned customers dominate the $0-500 range (~200 churned vs ~500 retained), reinforcing the early tenure churn pattern - customers leave before accumulating significant spend. As total charges increase beyond $2000, the proportion of retained customers increases substantially, with very few churned customers in the $6000+ range. This suggests that total invested spend creates a "sunk cost" retention effect.*

***Avg Monthly Long Distance Charges**: Exhibits extreme concentration at or near zero for both groups, with the vast majority of customers showing minimal long distance usage. Both churned (~600) and retained (~200) customers peak at the zero/near-zero range. A smaller segment shows distributed long distance charges from $5-50, but the proportions between churned and retained appear relatively consistent across these ranges. This feature shows limited discriminatory power - long distance usage doesn't appear to be a strong churn predictor, likely because it represents a small portion of most customers' bills.*

***Avg Monthly GB Download**: Shows a heavily right-skewed distribution with extreme concentration at zero/low usage for both groups. The majority of customers (both churned ~1,500 and retained ~400) have minimal data usage (0-10 GB). As data usage increases (10-80 GB), both groups show declining but proportionally similar frequency patterns. High data users (50+ GB) are rare in both segments. This suggests data usage intensity is not a strong churn predictor - both heavy and light users churn at similar rates, possibly indicating that churn is driven more by pricing/value perception than usage patterns.*

***Total Refunds**: Demonstrates an extremely stark concentration at zero for both groups, with the vast majority of customers (~4,500 retained, ~1,800 churned) having zero refunds. The distribution drops off precipitously - very few customers have any refunds at all, and those who do rarely exceed $5-10. This feature has minimal variance and therefore limited predictive value. The near-universal zero refund pattern suggests either: (1) refunds are rare events in this business, or (2) customers who experience service issues requiring refunds may already be captured in churn before refunds are processed.*

***Total Extra Data Charges**: Shows perhaps the most extreme concentration pattern of all features - virtually all customers (~4,500 retained, ~1,500 churned) cluster at zero extra data charges. The distribution is so heavily concentrated at zero that any non-zero charges are barely visible on the histogram. This indicates that extra data charges are extremely rare events, affecting only a tiny minority of customers. Given the lack of variance, this feature has negligible predictive power for churn modeling.*

***Total Revenue**: Displays a pattern very similar to Total Charges - exponential decay with heavy concentration in lower revenue ranges. Churned customers dominate the $0-1000 range (~400-700 frequency peaks), while retained customers show more substantial frequency extending into $2000-4000+ ranges. The separation becomes more pronounced at higher revenue levels - customers generating $4000+ in total revenue are overwhelmingly retained with minimal churn. This reinforces the customer lifetime value (CLV) concept: low-revenue customers are high-risk, while high-revenue customers demonstrate strong retention, likely due to both relationship investment and service dependency.*

**Critical Insights Across Continuous Service Features**:
1. **Tenure is the strongest predictor** - the first 0-10 months represent extreme vulnerability, with churn risk dropping dramatically after 12 months
2. **Financial investment creates stickiness** - Total Charges and Total Revenue show clear patterns where higher cumulative spend correlates with retention
3. **Monthly pricing sensitivity exists** - customers in the $65-100 monthly charge range show elevated churn, suggesting a competitive "shopping around" threshold
4. **Usage patterns (data, long distance) have minimal predictive power** - churn appears driven more by pricing/value/relationship factors than consumption behavior
5. **Refunds and extra charges are too rare** to serve as meaningful predictive features
6. **The "new customer danger zone"** - customers with low tenure, low total spend, and mid-to-high monthly charges represent the highest-risk profile

## 3F. Service Features (Ordinal)

### Purpose
Analyze ordinal service features (ordered categories) to understand how contract types, payment methods, internet types, and offers affect churn patterns.

### Key Questions Answered
- Do longer contracts reduce churn?
- Which payment methods are associated with lower churn?
- Does internet type (DSL vs Fiber) impact retention?
- How do different offer types affect churn rates?

### Insights to Gather
- Churn rates across each ordinal category
- Hierarchy of risk (which categories are most/least risky)
- Impact of contract commitment on retention
- Effectiveness of different promotional offers

### Key Features Analyzed
Ordinal service features:
- **Contract**: Month-to-Month vs One Year vs Two Year
- **PaymentMethod**: Bank Withdrawal vs Credit Card vs Mailed Check
- **InternetType**: No Internet vs DSL vs Cable vs Fiber Optic
- **Offer**: No Offer vs Offer A/B/C/D/E

### Churn Rates by Feature
**Contract:**
- Month-to-Month: 42.8% churn rate (1655/3869 customers)
- One Year: 10.7% churn rate (166/1558 customers)
- Two Years: 2.7% churn rate (48/1803 customers)

**PaymentMethod:**
- Bank Withdrawal: 34.0% churn rate (1329/3909 customers)
- Credit Card: 14.5% churn rate (398/2749 customers)
- Mailed Check: 36.9% churn rate (142/385 customers)

**InternetType:**
- No Internet: 7.4% churn rate (113/1526 customers)
- DSL: 18.8% churn rate (307/1632 customers)
- Cable: 25.7% churn rate (213/830 customers)
- Fiber Optic: 40.7% churn rate (1236/3035 customers)

**Offer:**
- No Offer: 27.1% churn rate (1051/3877 customers)
- Offer A: 6.7% churn rate (35/520 customers)
- Offer B: 12.3% churn rate (101/824 customers)
- Offer C: 22.9% churn rate (95/415 customers)
- Offer D: 26.7% churn rate (161/602 customers)
- Offer E: 53.6% churn rate (426/885 customers)

In [ ]:
# =============================================================================
# SECTION 3F: SERVICE FEATURES (ORDINAL)
# =============================================================================

print("=" * 80)
print("SECTION 3F: SERVICE FEATURES - ORDINAL")
print("=" * 80)

ordinal_features = {
    'Contract': {0: 'Month-to-Month', 1: 'One Year', 2: 'Two Year'},
    'PaymentMethod': {0: 'Bank Withdrawal', 1: 'Credit Card', 2: 'Mailed Check'},
    'InternetType': {0: 'No Internet', 1: 'DSL', 2: 'Cable', 3: 'Fiber Optic'},
    'Offer': {0: 'No Offer', 1: 'Offer A', 2: 'Offer B', 3: 'Offer C', 4: 'Offer D', 5: 'Offer E'}
}

# Statistical Analysis
print("\n📊 CHURN RATES BY ORDINAL FEATURES:")
print("-" * 70)

for feature, labels in ordinal_features.items():
    if feature in data.columns:
        churn_by_feature = data.groupby(feature)['Churn'].agg(['sum', 'count', 'mean'])
        print(f"\n{feature}:")
        for idx, row in churn_by_feature.iterrows():
            label = labels.get(idx, f'Unknown ({idx})')
            print(f"   {label}: {row['mean']:.1%} churn rate ({int(row['sum'])}/{int(row['count'])} customers)")

# Visualization: Ordinal Features Bar Charts
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Ordinal Service Features: Churn Distribution', fontsize=16, fontweight='bold')

for idx, (feature, labels) in enumerate(ordinal_features.items()):
    if feature in data.columns:
        row = idx // 2
        col = idx % 2

        # Get counts
        feature_churn = churn_data[feature].value_counts().sort_index()
        feature_nochurn = no_churn_data[feature].value_counts().sort_index()

        # Create DataFrame with correct labels
        max_idx = max(labels.keys())
        feature_df = pd.DataFrame({
            'No Churn': [feature_nochurn.get(i, 0) for i in range(max_idx + 1)],
            'Churn': [feature_churn.get(i, 0) for i in range(max_idx + 1)]
        }, index=[labels.get(i, f'Cat {i}') for i in range(max_idx + 1)])

        # Plot
        feature_df.plot(kind='bar', ax=axes[row, col], color=COLORS)
        axes[row, col].set_title(f'{feature} Distribution')
        axes[row, col].set_xlabel(feature)
        axes[row, col].set_ylabel('Number of Customers')
        axes[row, col].set_xticks(range(len(feature_df.index)))
        axes[row, col].set_xticklabels(feature_df.index, rotation=45, ha='right')
        axes[row, col].legend(title='Status')
        axes[row, col].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Section 3F Complete\n")

### Key Findings

***Contract Type**: Displays the most dramatic and clear ordinal relationship with churn of any feature analyzed. Month-to-Month contracts show catastrophic churn at 42.8% - nearly double the baseline rate of 26.5%. The customer base is almost evenly split between Month-to-Month (approximately 2000 retained, 1650 churned) and Two Year contracts (approximately 1800 retained, minimal churn). One Year contracts occupy a middle ground (approximately 1400 retained, 160 churned) with 10.7% churn. This creates a perfect ordinal progression: longer commitment = exponentially lower churn. The Two Year contract effectively "locks in" customers with only 2.7% churn, while Month-to-Month customers are essentially "at-risk by design" with the flexibility to leave at any time driving the extreme churn rate.*

***Payment Method**: Shows significant variation in churn rates, though the relationship is not perfectly ordinal. Bank Withdrawal (approximately 2600 retained, 1300 churned) and Mailed Check (approximately 240 retained, 140 churned) both show high churn rates around 34-37%, while Credit Card (approximately 2300 retained, 400 churned) demonstrates substantially lower churn at 14.5%. The pattern suggests that automatic electronic payment methods (bank withdrawal) don't necessarily prevent churn - customers can still cancel despite autopay. Credit Card users show the strongest retention, possibly indicating: (1) higher financial stability, (2) rewards/points motivation to maintain payment method, or (3) demographic correlation with more engaged customers. Mailed Check's high churn rate may reflect older, less tech-savvy customers who are less engaged with modern service options.*

***Internet Type**: Reveals a clear and concerning ordinal pattern that directly contradicts typical business logic. No Internet customers (approximately 1400 retained, 100 churned) show exceptional retention at 7.4% churn. However, as internet service tier increases, churn escalates dramatically: DSL 18.8%, Cable 25.7%, and Fiber Optic reaches a staggering 40.7% churn (approximately 1800 retained, 1200 churned). Fiber Optic, typically the premium, highest-speed offering, shows the worst retention of any internet type. This counterintuitive finding suggests: (1) Fiber customers are in highly competitive markets with multiple provider options, (2) Premium customers are more price-sensitive and actively shop for better deals, (3) Fiber service may have quality/reliability issues not captured in other metrics, or (4) Higher expectations among premium customers lead to greater dissatisfaction. The progression indicates that upgrading customers to premium services without addressing underlying value/competition issues actually increases churn risk.*

***Offer Type**: Demonstrates a complex pattern where promotional offers show highly variable effectiveness. No Offer customers (approximately 2800 retained, 1050 churned) sit near baseline at 27.1% churn. Offers A and B show strong protective effects (6.7% and 12.3% churn respectively), suggesting these are well-designed retention offers. However, Offers C and D (22.9% and 26.7% churn) provide minimal benefit over no offer at all. Most alarmingly, Offer E shows catastrophic 53.6% churn (approximately 450 retained, 425 churned) - double the baseline rate and worse than having no offer. This suggests Offer E may be a "last-ditch" retention attempt deployed to already-decided churners, or it's poorly structured and actually signals to customers that they should leave. The visualization shows that simply having "an offer" is not protective - offer quality and targeting matter enormously.*

**Critical Insights Across Ordinal Service Features:**
1. **Contract commitment is the single strongest churn preventer** - Two Year contracts reduce churn to 2.7% vs 42.8% for Month-to-Month, a 15.8x difference
2. **Premium internet service paradoxically increases churn risk** - Fiber Optic customers churn at 40.7% despite being highest-paying segment
3. **Payment method matters, but not as expected** - Credit Card shows best retention (14.5%) while Bank Withdrawal autopay still churns at 34.0%
4. **Promotional offers can backfire** - Offer E at 53.6% churn suggests poor targeting or offer design can actively harm retention
5. **The "bare-bones but locked in" profile wins** - Customers with long contracts and basic service (No Internet or DSL) show the strongest retention patterns
6. **High-value customers are flight risks** - The combination of Month-to-Month contracts, Fiber internet, and certain offers creates a perfect storm for churn exceeding 50%


# Section 4: Correlation Analysis

## Purpose
Identify and quantify the relationships between features and churn, as well as inter-feature correlations, to inform feature selection and understand multicollinearity for predictive modeling.

## Key Questions Answered
- Which features have the strongest correlation with churn?
- Are there redundant features that are highly correlated with each other?
- What are the key positive and negative correlations with churn?
- Which features can be safely excluded due to weak correlation?

## Insights to Gather
- Top positive correlations with churn (churn risk factors)
- Top negative correlations with churn (protective factors)
- Feature multicollinearity patterns
- Feature importance ranking based on correlation strength

## Top 15 Positive Correlations with Churn

1. **InternetType**: 0.383 - Strongest positive correlation, confirming that internet service type (particularly Fiber Optic) is the primary churn driver
2. **InternetService**: 0.228 - Having internet service itself positively correlates with churn
3. **PaperlessBilling**: 0.192 - Paperless billing customers show higher churn tendency
4. **MonthlyCharge**: 0.169 - Higher monthly charges associated with increased churn
5. **UnlimitedData**: 0.167 - Unlimited data plans correlate with higher churn
6. **Offer**: 0.120 - Certain promotional offers show positive churn correlation
7. **Age**: 0.116 - Slight positive correlation with older customers
8. **StreamingTV**: 0.063 - Minimal positive correlation
9. **StreamingMovies**: 0.061 - Minimal positive correlation
10. **Population**: 0.052 - Very weak positive correlation
11. **AvgMonthlyGBDownload**: 0.049 - Data usage shows negligible correlation
12. **StreamingMusic**: 0.046 - Minimal positive correlation
13. **MultipleLines**: 0.020 - Nearly no correlation
14. **PhoneService**: 0.012 - Virtually no correlation
15. **AvgMonthlyLongDistanceCharges**: 0.008 - No meaningful correlation

## Top 15 Negative Correlations with Churn

1. **Contract**: -0.435 - Strongest negative correlation by far, confirming contract length is the most powerful retention factor
2. **TenureInMonths**: -0.353 - Second strongest predictor, longer tenure dramatically reduces churn
3. **NumberOfReferrals**: -0.287 - Customers who refer others are significantly less likely to churn
4. **TotalLongDistanceCharges**: -0.224 - Higher accumulated long distance spend indicates loyalty
5. **TotalRevenue**: -0.223 - Higher total revenue generated correlates with retention
6. **NumberOfDependents**: -0.219 - Customers with dependents show better retention
7. **TotalCharges**: -0.215 - Higher total charges (longer relationship) reduces churn
8. **OnlineSecurity**: -0.171 - Security service adoption is protective
9. **PremiumTechSupport**: -0.165 - Premium support subscribers are more loyal
10. **Married**: -0.150 - Married customers show better retention
11. **PaymentMethod**: -0.135 - Certain payment methods (Credit Card) reduce churn
12. **OnlineBackup**: -0.082 - Backup service provides modest protection
13. **DeviceProtectionPlan**: -0.066 - Device protection shows weak protective effect
14. **TotalRefunds**: -0.034 - Minimal correlation
15. **Gender**: -0.009 - No meaningful correlation


In [ ]:
# =============================================================================
# Section 4: CORRELATION ANALYSIS
# =============================================================================

print("=" * 80)
print("CORRELATION ANALYSIS")
print("=" * 80)

# Calculate correlation matrix
print("\n📊 TOP FEATURES CORRELATED WITH CHURN:")
print("-" * 70)

# Select numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if 'Churn' in numeric_cols:
    corr_matrix = data[numeric_cols].corr()
    churn_corr = corr_matrix['Churn'].sort_values(ascending=False)

    # Display top 15 positive and negative correlations
    print("\nTop 15 Positive Correlations with Churn:")
    for feature, corr_value in churn_corr[1:16].items():
        print(f"   {feature}: {corr_value:.3f}")

    print("\nTop 15 Negative Correlations with Churn:")
    for feature, corr_value in churn_corr[-15:].sort_values().items():
        print(f"   {feature}: {corr_value:.3f}")

# Visualization: Correlation Heatmap
plt.figure(figsize=(16, 14))
corr_matrix_top = data[churn_corr.head(20).index.tolist()].corr()
mask = np.triu(np.ones_like(corr_matrix_top, dtype=bool))
sns.heatmap(corr_matrix_top, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title('Top 20 Features Correlation Matrix with Churn', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## Key Findings

*Top Churn Risk Factors: The correlation analysis confirms that internet service characteristics dominate churn risk. InternetType (0.383) and InternetService (0.228) are the two strongest positive correlations, followed by billing and pricing factors (PaperlessBilling 0.192, MonthlyCharge 0.169). This creates a clear risk profile: customers with premium internet service (especially Fiber), paperless billing, and higher monthly charges are most likely to churn. Notably, usage-based features (streaming services, data download, long distance) show very weak positive correlations (0.01-0.08 range), confirming that churn is driven by service/pricing perception rather than actual usage behavior.*

*Top Protective Factors: Contract commitment (-0.435) is the single most powerful anti-churn factor, nearly 25% stronger than the next factor. Tenure (-0.353) and Referrals (-0.287) form the "loyalty triad" - customers who commit to contracts, stay longer, and refer others are dramatically less likely to churn. Revenue-related metrics (TotalRevenue -0.223, TotalCharges -0.215) and relationship factors (NumberOfDependents -0.219, Married -0.150) form a secondary protective layer. Security/support services show moderate negative correlations (-0.17 to -0.08), confirming their retention value but suggesting they're not as powerful as fundamental relationship factors.*

*Feature Multicollinearity Patterns: The correlation matrix reveals several concerning multicollinearity clusters. InternetType and InternetService show strong correlation (0.76), suggesting potential redundancy. MonthlyCharge shows high correlation with both InternetService (0.83) and InternetType (0.72), indicating that internet service tier is the primary driver of monthly pricing. Streaming services (TV, Movies, Music) are highly intercorrelated (0.53-0.85), suggesting they could be combined into a single "streaming adoption" feature. TotalCharges and TotalRevenue show near-perfect correlation (implied but not directly shown), indicating one could be excluded. AvgMonthlyLongDistanceCharges correlates strongly with TotalLongDistanceCharges (0.49), creating redundancy in long distance metrics.*

*Weak and Irrelevant Features: Several features show negligible correlation with churn and could be candidates for exclusion. Gender (-0.009) is essentially uncorrelated with churn, confirming demographic gender has no predictive value. TotalRefunds (-0.034), TotalExtraDataCharges (implied near-zero), and Population (0.052) all show minimal correlation, suggesting these features add more noise than signal. PhoneService (0.012) and AvgMonthlyLongDistanceCharges (0.008) are near-zero, indicating phone-related metrics have minimal churn predictive power compared to internet-related features.*

*Correlation Magnitude Insights: The correlation values reveal a clear hierarchy of predictive power. Strong correlations (|r| > 0.3) are limited to Contract, Tenure, InternetType, and NumberOfReferrals. Moderate correlations (0.15 < |r| < 0.3) include InternetService, financial metrics (TotalRevenue, TotalCharges), and relationship factors (NumberOfDependents, Married). Weak correlations (|r| < 0.15) dominate the feature space, including most service features, usage metrics, and demographics. This suggests that a small subset of features (5-10) likely captures most of the predictive signal, while the majority contribute minimal incremental value.*

*Risk vs Protection Asymmetry: An interesting pattern emerges when comparing positive and negative correlations. The strongest churn risk factor (InternetType 0.383) is weaker than the strongest protective factor (Contract -0.435), suggesting it's easier to prevent churn through contracts than it is to cause churn through internet service type alone. However, the positive correlations are more numerous in the moderate range (5 features >0.15) compared to negative correlations (3 features <-0.15), suggesting that while strong protection is concentrated in a few factors, churn risk is more distributed across multiple service characteristics.*

**Critical Pattern for Feature Engineering**: The correlation analysis suggests an optimal feature reduction strategy: (1) Keep Contract, Tenure, NumberOfReferrals as core relationship features, (2) Consolidate internet features (InternetType/InternetService/MonthlyCharge) into composite metrics, (3) Create a single "protective services bundle" feature combining OnlineSecurity/PremiumTechSupport/OnlineBackup rather than treating them separately, (4) Eliminate near-zero correlation features (Gender, PhoneService, Refunds, ExtraDataCharges), and (5) Address streaming service multicollinearity by creating a streaming service count or adoption score rather than separate binary features.

## Summary

The correlation analysis reveals a clear hierarchy of churn predictors:

**Tier 1 Predictors (|r| > 0.3):**
- Contract commitment and Tenure are the dominant protective factors
- InternetType is the primary risk factor

**Tier 2 Predictors (0.15 < |r| < 0.3):**
- Financial investment metrics (TotalRevenue, TotalCharges)
- Relationship factors (NumberOfReferrals, NumberOfDependents, Married)
- Service characteristics (InternetService, PaperlessBilling, MonthlyCharge)

**Tier 3 and Below (|r| < 0.15):**
- Most service features, usage metrics, and demographics show weak or negligible correlations
- Many features are candidates for elimination or consolidation

**Key Takeaway**: A small number of features (Contract, Tenure, InternetType, NumberOfReferrals, financial metrics) capture the majority of churn signal, while most features contribute minimal incremental predictive value. Feature engineering should focus on consolidating redundant features and eliminating noise while preserving the core predictive signals.